In [3]:
import pandas as pd
import sklearn as skl
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import recall_score, precision_score, plot_confusion_matrix, f1_score, ConfusionMatrixDisplay
import seaborn as sns
import numpy as np
from imblearn.over_sampling import RandomOverSampler

In [5]:
main_df = pd.read_csv('./raisin-training.csv')
holdout = pd.read_csv('./raisin-holdout.csv')

In [37]:
main_df.Class.unique()
# holdout.describe()

array(['Kecimen', 'Besni'], dtype=object)

In [6]:
main_dummy_df = pd.get_dummies(main_df, drop_first=True)
main_dummy_df = main_dummy_df.rename(columns={"Class_Kecimen": "Class"})
main_dummy_df

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,97026,455.971591,273.053810,0.800869,99561,0.671205,1212.667,1
1,65253,418.997887,205.756185,0.871122,69700,0.666255,1075.404,0
2,113029,558.516156,265.284203,0.879996,116783,0.662092,1419.577,0
3,76792,338.857545,291.359202,0.510584,78842,0.772322,1042.770,1
4,72219,376.650492,249.529454,0.749065,74373,0.777795,1050.221,1
...,...,...,...,...,...,...,...,...
693,46742,303.555203,199.445933,0.753861,48077,0.705263,847.792,1
694,55787,333.703453,226.951208,0.733121,59520,0.688592,977.425,1
695,61996,333.747640,243.540245,0.683753,63641,0.673138,958.627,1
696,74728,355.310549,270.740897,0.647596,76287,0.766677,1048.675,1


In [129]:

dtc = RandomForestClassifier(max_samples=2, class_weight='balanced_subsample', \
    max_features=None, random_state=2)
# dtc = RandomForestClassifier(class_weight='balanced_', random_state=11)

X = main_dummy_df.drop(['Class'],axis = 1)
y = main_dummy_df['Class']

ros = RandomOverSampler(random_state=11)
xee, yee = ros.fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(xee, yee, test_size = 0.5, random_state = 42)
X_val, _, y_val, _ = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42)

dtc.fit(X_train, y_train)

score = dtc.score(X_val, y_val)
y_pred = dtc.predict(X_val)
recall = recall_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
score, recall, precision, len(y_pred)

# pred_df = pd.DataFrame()
# pred_df['prediction'] = y_pred
# pred_df.to_csv('main_holdout_predictions.csv', index=False)
# con_mat = plot_confusion_matrix(dtc, X_test, y_test, cmap='flare_r')

(0.8571428571428571, 0.8229166666666666, 0.9080459770114943, 175)

In [127]:
holdout_predictions = dtc.predict(holdout)
holdout_predictions = pd.DataFrame(holdout_predictions)
holdout_predictions = holdout_predictions.rename(columns={0: 'Class'})
holdout_predictions.to_csv('holdout_encoded_guesses.csv', index=None)
